# Feature Engineering

In this notebook, we will perform feature engineering on the customer dataset. This includes creating the RFM (Recency, Frequency, Monetary) table, handling outliers, scaling features, and saving the processed data.

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Load the raw online retail data
data = pd.read_csv('../data/raw/online_retail.csv', encoding='latin-1')

# Create TotalSpent column if it doesn't exist
if 'TotalSpent' not in data.columns:
    data['TotalSpent'] = data['Quantity'] * data['UnitPrice']

# Convert InvoiceDate to datetime
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'])

# Remove rows with missing CustomerID
data = data.dropna(subset=['CustomerID'])

# Display the first few rows
print(f"Data shape: {data.shape}")
data.head()

Data shape: (406829, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalSpent
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


## Creating the RFM Table

We will create the RFM table to segment customers based on their purchasing behavior.

In [5]:
# Create RFM table
# Set snapshot date as one day after the most recent transaction
snapshot_date = data['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = data.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'count',  # Frequency
    'TotalSpent': 'sum'  # Monetary
}).rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalSpent': 'Monetary'
})

# Reset index to make CustomerID a column
rfm = rfm.reset_index()

print(f"RFM table shape: {rfm.shape}")
print(f"\nRFM Statistics:")
print(rfm.describe())
print(f"\nFirst few rows:")
rfm.head()

RFM table shape: (4372, 4)

RFM Statistics:
         CustomerID      Recency    Frequency       Monetary
count   4372.000000  4372.000000  4372.000000    4372.000000
mean   15299.677722    92.047118    93.053294    1898.459701
std     1722.390705   100.765435   232.471608    8219.345141
min    12346.000000     1.000000     1.000000   -4287.630000
25%    13812.750000    17.000000    17.000000     293.362500
50%    15300.500000    50.000000    42.000000     648.075000
75%    16778.250000   143.000000   102.000000    1611.725000
max    18287.000000   374.000000  7983.000000  279489.020000

First few rows:


,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,2,0.00
1,12347.0,2,182,4310.00
2,12348.0,75,31,1797.24
3,12349.0,19,73,1757.55
4,12350.0,310,17,334.40


## Handling Outliers

Next, we will handle outliers in the Monetary value to ensure that our analysis is not skewed.

In [6]:
# Identify outliers using IQR method
Q1 = rfm['Monetary'].quantile(0.25)
Q3 = rfm['Monetary'].quantile(0.75)
IQR = Q3 - Q1
rfm = rfm[~((rfm['Monetary'] < (Q1 - 1.5 * IQR)) | (rfm['Monetary'] > (Q3 + 1.5 * IQR)))]

# Display the RFM table after outlier removal
rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,2,0.00
2,12348.0,75,31,1797.24
3,12349.0,19,73,1757.55
4,12350.0,310,17,334.40
5,12352.0,36,95,1545.41


## Scaling Features

We will scale the RFM features to prepare them for clustering.

In [7]:
# Scale the RFM features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

# Convert scaled data back to DataFrame
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=rfm.columns)
rfm_scaled_df.head()

,CustomerID,Recency,Frequency,Monetary
0,-1.750560,2.210736,-0.777296,-1.050747
1,-1.749388,-0.238759,-0.401135,1.157604
2,-1.748802,-0.785260,0.143651,1.108835
3,-1.748217,2.054593,-0.582730,-0.639855
4,-1.747045,-0.619358,0.429015,0.848168


## Saving the Processed Data

Finally, we will save the processed RFM data for use in the modeling phase.

In [8]:
# Save the processed RFM data
# Add CustomerID back to the scaled dataframe
rfm_scaled_df['CustomerID'] = rfm['CustomerID'].values

# Save to CSV
rfm_scaled_df.to_csv('../data/processed/rfm_scaled.csv', index=False)

print('✅ Processed RFM data saved successfully!')
print(f'Shape: {rfm_scaled_df.shape}')
print(f'Columns: {rfm_scaled_df.columns.tolist()}')

✅ Processed RFM data saved successfully!
Shape: (3949, 4)
Columns: ['CustomerID', 'Recency', 'Frequency', 'Monetary']
